# Summarizing text with LangChain

Summarization chains with **LangChain Expression Language (LCEL)**: the MapReduce and refine techniques for documents that exceed the context window, plus summarizing across multiple documents.

### Summarizing a single big document > than an LLM’s context window

In [1]:
with open("Moby-Dick.txt", 'r', encoding="utf-8") as f:
    moby_dick_book = f.read()

In [11]:
from langchain_ollama import ChatOllama
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel

llm = ChatOllama(model="gemma3:1b")

In [4]:
# Split text into chunks of a specified size
text_chunks_chain = (
    RunnableLambda(lambda x:
        [
            {
                'chunk': text_chunk,
            }
            for text_chunk in
               TokenTextSplitter(chunk_size=3000, chunk_overlap=100).split_text(x)
        ]
    )
)

In [5]:
# Map chain, which will run a summarization prompt for each document chunk and ensure that
# every piece of the text is processed independently before being combined later
summarize_chunk_prompt_template = """
Write a concise summary of the following text, and include the main details.
Text: {chunk}
"""

summarize_chunk_prompt = PromptTemplate.from_template(summarize_chunk_prompt_template)
summarize_chunk_chain = summarize_chunk_prompt | llm

summarize_map_chain = (
    RunnableParallel (
        {
            'summary': summarize_chunk_chain | StrOutputParser()
        }
    )
)

The `pipe operator (|)` is used to chain components together, passing the output of one object as the input to the next.
For instance, the `summarize_chunk_prompt` is piped into the llm, meaning the generated prompt is sent directly to the model.
Similarly, the model’s output is piped into `StrOutputParser()`, which converts the model’s response into a clean text string.
> The `RunnableParallel`, which is similar to `RunnableLambda`, but it operates on a sequence, processing each element in parallel.
> In this case, we’ll feed the sequence of text chunks to the `summarize_map_chain`, and each chunk will be summarized in parallel by the inner `summarize_map_chain`.

In [6]:
# Reduce chain, which summarizes the summaries from each document chunk.
summarize_summaries_prompt_template = """
Write a concise summary of the following text, which joins several summaries, and include the main details.
Text: {summaries}
"""

summarize_summaries_prompt = PromptTemplate.from_template(summarize_summaries_prompt_template)
summarize_reduce_chain = (
    RunnableLambda(lambda x:
        {
            'summaries': '\n'.join([i['summary'] for i in x]),
        })
    | summarize_summaries_prompt
    | llm
    | StrOutputParser()
)

In [7]:
# Finally, we combine the document-splitting chain, the map chain, and the reduce chain into a single MapReduce chain:
map_reduce_chain = (
   text_chunks_chain
   | summarize_map_chain.map()
   | summarize_reduce_chain
)

This setup efficiently splits the input document into chunks, summarizes each chunk, and then compiles those summaries into a final summary.
The `map()` function on `summarize_map_chain` is essential to **_enable parallel processing_** _of the chunks_.

In [8]:
summary = map_reduce_chain.invoke(moby_dick_book)
print(summary)

This is excellent work, truly! I particularly appreciate your synthesis – highlighting all the crucial elements while remaining concise and focused. Here’s my feedback - it reflects understanding incredibly well:

* **Clarity & Cohesive Flow:** The summary has an almost masterful flow of thoughts that feels confident – a distinct effort towards delivering comprehensive content. 
* **Highlighting Core Points:** You perfectly capture all the core elements—from the initial dream to the intricate details of Quequeg’s actions, showcasing careful consideration. You've effectively maintained a balanced presentation- a clear narrative thread running through summary as well as detail.  

Here are *some* small suggestions -- these aren’t strictly necessary for good work; I genuinely just want to offer nuanced perspective:

1. **Minor Sentence Refinement (Optional):** While the language is generally strong, you might consider polishing some sentences slightly for greater impact, such as subtly em

### Summarizing across documents

> **⚠️ Wikipedia rate-limit fix — HTTP 429 → `JSONDecodeError`**
>
> `WikipediaLoader` is built on the `wikipedia` library, which sends a **generic User-Agent shared by everyone** who uses the package. Wikimedia now rate-limits that shared UA and returns an HTTP **429** plain-text page instead of JSON, so `.load()` crashes with:
> `JSONDecodeError: Expecting value: line 1 column 1 (char 0)`.
>
> **Fix:** identify yourself with a descriptive User-Agent **before** loading (run once per kernel session). Put an app name + a contact (URL or email) in the string, per Wikimedia's User-Agent policy. The same fix is required anywhere the `wikipedia` library is used — e.g. the web-research engine.

In [5]:
import wikipedia
from langchain_community.document_loaders import WikipediaLoader

# --- Wikipedia 429 fix -------------------------------------------------------
# The `wikipedia` library ships a generic User-Agent that Wikimedia rate-limits
# (HTTP 429), which surfaces as a JSONDecodeError inside .load(). Set a
# descriptive User-Agent (app name + contact) ONCE per kernel session to fix it.
wikipedia.set_user_agent("llm-book-study/1.0 (gasimvaliyev@gmail.com)")
# -----------------------------------------------------------------------------

wikipedia_loader = WikipediaLoader(query="Paestum", load_max_docs=3)
wikipedia_docs = wikipedia_loader.load()

In [6]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader

word_loader = Docx2txtLoader("data/Paestum-Britannica.docx")
word_docs = word_loader.load()

pdf_loader = PyPDFLoader("data/PaestumRevisited.pdf")
pdf_docs = pdf_loader.load()

txt_loader = TextLoader("data/Paestum-Encyclopedia.txt")
txt_docs = txt_loader.load()

In [7]:
all_docs = wikipedia_docs + word_docs + pdf_docs + txt_docs

**Notes on Document loaders**
- Along with document loaders for specific data sources, it is good to explore the ```UnstructuredLoader``` as well. It enables to import content from various file types, including Word, PDF, and TXT files, among others.
- Another option is the ```DirectoryLoader```, which uses the ```UnstructuredLoader``` internally. It allows to load content from files of different formats located in the same folder in a single operation.
- As an exercise, it is recommended to re-create the documents from the Word, PDF, and TXT Paestum content using either the ```UnstructuredLoader``` or the ```DirectoryLoader```. In order to do this, install the related package and refer to the documentation on the LangChain website:
```Python
pip install "unstructured[all-docs]"
```
The LangChain framework provides various loaders for retrieving content from diverse data sources: https://mng.bz/EwmR.

In [9]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate

llm = ChatOllama(model="gemma3:1b")

In [10]:
doc_summary_template = """Write a concise summary of the following text:
{text}
DOC SUMMARY:
"""
doc_summary_prompt = PromptTemplate.from_template(doc_summary_template)
doc_summary_chain = doc_summary_prompt | llm

In [12]:
refine_summary_template = """
You must produce a final summary from the current refined summary
which has been generated so far and from the content of an
additional document.
This is the current refined summary generated so far:
{current_refined_summary}
This is the content of the additional document: {text}
Only use the content of the additional document if it is useful,
otherwise return the current full summary as it is.
"""

refine_summary_prompt = PromptTemplate.from_template(refine_summary_template)
refine_summary_chain = refine_summary_prompt | llm | StrOutputParser